In [21]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_timestamp, lower, current_timestamp, concat, lpad, lit, when, trim, substring
from pyspark.sql.types import StringType, BooleanType
import re
import time
import json

#The second line (with 4g) specifies how much RAM to use. change according to machine
spark = SparkSession.builder \
    .appName("IngestionFramework") \
    .config("spark.driver.memory", "4g") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()



# DATASET_CONFIG is a json file 
# Has entries that are specific to each dataset, like:
# primary_keys: if its a list then its the combination of these columns
# timestamp_cols: dict with columns cast to TimestampType (after snake_case rename).
# assembled_timestamp : dict with keys year_col, month_col, day_col,
#                         and optionally hour_col, minute_col, second_col,
#                         plus output_col for the new timestamp column name.
#                         Used when date parts live in separate integer columns.
# date_time_pairs     : list of {date_col, time_col, output_col, format}
#                         Used when date and time live in separate string columns.
# bool_cols           : list of column names to cast to BooleanType.
#                         Understands Y/N, yes/no, true/false, 1/0.
with open('dataset_config.json', 'r') as file:
    DATASET_CONFIGS = json.load(file)


    TYPE_MAP = {
        "integer": "int",
        "int": "int",
        "long": "bigint",
        "bigint": "bigint",
        "float": "float",
        "double": "double",
        "string": "string",
        "boolean": "boolean",
        "bool": "boolean",
        "timestamp": "timestamp",
        "date": "date",
    }

#-----------------------------Helpers-------------------

#column name to snake case

def to_snake_case(name):
    s = name.strip()
    s = re.sub(r'[\s\-]+', '_', s)                     # Replace spaces/dashes with single underscore first
    s1 = re.sub(r'([A-Z]+)([A-Z][a-z])', r'\1_\2', s)   # Handle runs of caps (e.g. PULocation -> PU_Location)
    s2 = re.sub(r'([a-z0-9])([A-Z])', r'\1_\2', s1)    # Handle lower to upper transitions
    s3 = re.sub(r'_+', '_', s2)                        # Collapse any consecutive underscores
    return s3.lower()

#also snake case function
def standardize_columns(df):
    for c in df.columns:
        df = df.withColumnRenamed(c, to_snake_case(c))
    return df


#validate if the rows in dataset are the same as expected_rows in config file (same name and type)
def validate_input_schema(df, config, dataset_name):
    errors = []
    actual_types = dict(df.dtypes)
    
    #validate column presence
    if "expected_columns" in config:
        expected_cols = {to_snake_case(c) for c in config["expected_columns"]}
        actual_cols = set(df.columns)
        
        if missing := expected_cols - actual_cols:
            errors.append(f"Missing columns: {sorted(missing)}")
        if extra := actual_cols - expected_cols:
            errors.append(f"Extra columns: {sorted(extra)}")

    # validate data type
    for col_name, exp_type in config.get("schema", {}).items():
        act_type = actual_types.get(col_name)
        if not act_type:
            continue 
            
        
        exp_norm = TYPE_MAP.get(exp_type.lower(), exp_type.lower())
        act_norm = TYPE_MAP.get(act_type.lower(), act_type.lower())
        
        if exp_norm != act_norm:
            errors.append(f"Type mismatch '{col_name}': expected '{exp_norm}', got '{act_norm}'")

    if errors:
        raise ValueError(f"[{dataset_name}] Schema validation failed:\n - " + "\n - ".join(errors))

    
# Cast each column to the type declared in config["schema"]
def apply_schema(df, config):
    declared_schema = config.get("schema")
    if not declared_schema:
        return df
    for col_name, spark_type in declared_schema.items():
        if col_name in df.columns:
            df = df.withColumn(col_name, col(col_name).cast(spark_type))
    return df


# Timestamp normalisation to format yyyy-mm-dd HH:mm:ss
#3 types of normalization

# Cast timestamp to TimeStampType
def normalize_timestamps(df, timestamp_cols):
    
    for col_name, fmt in timestamp_cols.items():
        snake = to_snake_case(col_name)
        # If the column is already a TimestampType (e.g. from Parquet), keep it;
        # otherwise parse using the supplied format string.
        col_dtype = dict(df.dtypes).get(snake)
        if col_dtype == "timestamp":
            pass  
        else:
            df = df.withColumn(snake, to_timestamp(col(snake), fmt))
    return df


# Build a single TimestampType column from separate integer or string
# year / month / day / [hour / minute / second] columns.
# Used for weather dataset
def assemble_timestamp_from_parts(df, config):
    if not config:
        return df

    year   = config["year_col"]
    month  = config["month_col"]
    day    = config["day_col"]
    hour   = config.get("hour_col")
    minute = config.get("minute_col")
    second = config.get("second_col")
    out    = config["output_col"]

    # Build a string like "2024-01-01 00:00:00" then parse it
    time_part = concat(
        lpad(col(hour).cast("string"),   2, "0") if hour   else lit("00"), lit(":"),
        lpad(col(minute).cast("string"), 2, "0") if minute else lit("00"), lit(":"),
        lpad(col(second).cast("string"), 2, "0") if second else lit("00"),
    )
    date_part = concat(
        col(year).cast("string"),  lit("-"),
        lpad(col(month).cast("string"), 2, "0"), lit("-"),
        lpad(col(day).cast("string"),   2, "0"),
    )
    datetime_str = concat(date_part, lit(" "), time_part)
    df = df.withColumn(out, to_timestamp(datetime_str, "yyyy-MM-dd HH:mm:ss"))
    return df

# Combine separate date-string and time-string columns into a single column
# Used for air_quality dataset
def normalize_date_time_pairs(df, pairs):
    for pair in pairs:
        date_col   = pair["date_col"]
        time_col   = pair["time_col"]
        output_col = pair["output_col"]
        fmt        = pair["format"]
        datetime_str = concat(col(date_col), lit(" "), col(time_col))
        df = df.withColumn(output_col, to_timestamp(datetime_str, fmt))
    return df


# Common data type normalisation

# Formatting (trim whitespace, empty places become NULL)
def normalize_string_columns(df):
    for field in df.schema.fields:
        if isinstance(field.dataType, StringType):
            c = field.name
            df = df.withColumn(c, when(trim(col(c)) == "", None).otherwise(trim(col(c))))
    return df

# Replace boolean-like columns with actual boolean (Y / N become true and falls, etc...)
def normalize_boolean_columns(df, bool_cols):
    truthy = {"y", "yes", "true", "1"}
    falsy  = {"n", "no",  "false", "0"}

    for c in bool_cols:
        if c not in df.columns:
            continue
        lowered = lower(trim(col(c).cast("string")))
        df = df.withColumn(
            c,
            when(lowered.isin(*truthy), lit(True))
            .when(lowered.isin(*falsy),  lit(False))
            .otherwise(None)
            .cast(BooleanType())
        )
    return df

#Removes duplicates according to PK (or full row duplicates)
def perform_data_quality_checks(df, primary_keys):
    initial_count = df.count()

    if primary_keys:
        # Drop row if ANY part of the composite primary key is missing (Null)
        df = df.dropna(subset=primary_keys)

        # Drop duplicates based strictly on the combination of the primary keys
        df = df.dropDuplicates(subset=primary_keys)
    else:
        # If no primary key exists (like Taxi Trips), just drop exact full-row duplicates
        df = df.dropDuplicates()

    final_count = df.count()
    rejected_count = initial_count - final_count

    return df, initial_count, rejected_count


# For versioning in metadata
def schema_hash(df):
    return hash(tuple(sorted(df.dtypes)))


#-----------------------------Main Function-------------------
#steps follow what the assignment mentioned. 

def ingest_dataset(dataset_name, config):
    start_time = time.time()

    # Load according to file type (csv, parquet)
    df = spark.read.format(config["format"]).options(**config["options"]).load(config["path"])

    # Standardise column names (snake_case)
    df = standardize_columns(df)

    # cast each column to its expected type.
    df = apply_schema(df, config)

    # Validate input schema: column presence + type conformance
    validate_input_schema(df, config, dataset_name)

    # Normalise common data types (remove whitespace)
    df = normalize_string_columns(df)

    #  Normalise timestamps to yyyy-mm-dd HH:mm:ss
    df = normalize_timestamps(df, config.get("timestamp_cols", {}))
    #(weather)
    df = assemble_timestamp_from_parts(df, config.get("assembled_timestamp"))
    #(air quality)
    df = normalize_date_time_pairs(df, config.get("date_time_pairs", []))

    # Normalise boolean data types
    df = normalize_boolean_columns(df, config.get("bool_cols", []))

    # quality checks (removing duplicates)
    standard_pks = [to_snake_case(c) for c in config["primary_keys"]]
    df, initial_cnt, rejected_cnt = perform_data_quality_checks(df, standard_pks)

    # Write to Delta
    output_path = f"./delta/{dataset_name}"
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(output_path)


    execution_time = time.time() - start_time

    metadata = {
        "dataset":                dataset_name,
        "processed_records":      initial_cnt,
        "rejected_records":       rejected_cnt,
        "final_records":          initial_cnt - rejected_cnt,
        "execution_time_seconds": round(execution_time, 2),
        "schema_version":         schema_hash(df),
    }
    return metadata

#-----------------------------RUN framework-------------------
ingestion_logs = []
for name, conf in DATASET_CONFIGS.items():
    print(f"Ingesting {name}...")
    log = ingest_dataset(name, conf)
    ingestion_logs.append(log)

print(*ingestion_logs, sep="\n")


Ingesting weather...
Ingesting taxi_trips_01...


Ingesting taxi_trips_02...


Ingesting taxi_trips_03...


Ingesting taxi_zone_lookup...
Ingesting air_quality...


{'dataset': 'weather', 'processed_records': 8784, 'rejected_records': 0, 'final_records': 8784, 'execution_time_seconds': 2.07, 'schema_version': 8844775470973564984}
{'dataset': 'taxi_trips_01', 'processed_records': 2964624, 'rejected_records': 0, 'final_records': 2964624, 'execution_time_seconds': 6.78, 'schema_version': 5990212767973237942}
{'dataset': 'taxi_trips_02', 'processed_records': 3007526, 'rejected_records': 1, 'final_records': 3007525, 'execution_time_seconds': 6.83, 'schema_version': 5990212767973237942}
{'dataset': 'taxi_trips_03', 'processed_records': 3582628, 'rejected_records': 0, 'final_records': 3582628, 'execution_time_seconds': 7.85, 'schema_version': 5990212767973237942}
{'dataset': 'taxi_zone_lookup', 'processed_records': 265, 'rejected_records': 0, 'final_records': 265, 'execution_time_seconds': 1.11, 'schema_version': 2617290461417661303}
{'dataset': 'air_quality', 'processed_records': 8139551, 'rejected_records': 0, 'final_records': 8139551, 'execution_time_

In [24]:
df_delta1 = spark.read.format("delta").load("delta/taxi_trips_01")
df_delta2 = spark.read.format("delta").load("delta/weather")
df_delta3 = spark.read.format("delta").load("delta/taxi_zone_lookup")
df_delta4 = spark.read.format("delta").load("delta/air_quality")

df_delta1.show(1)
df_delta2.show(1)
df_delta3.show(1)
df_delta4.show(1)

+----------+-----------+--------+--------------+---+--------+----------+-----+--------------------+----------+----------+----------+--------+------------------+--------------------+---+-----------+---------+-----------+-----------+--------------------+----------+-----------+-------------------+-------------------+
|state_code|county_code|site_num|parameter_code|poc|latitude| longitude|datum|      parameter_name|date_local|time_local|  date_gmt|time_gmt|sample_measurement|    units_of_measure|mdl|uncertainty|qualifier|method_type|method_code|         method_name|state_name|county_name|date_of_last_change|     datetime_local|
+----------+-----------+--------+--------------+---+--------+----------+-----+--------------------+----------+----------+----------+--------+------------------+--------------------+---+-----------+---------+-----------+-----------+--------------------+----------+-----------+-------------------+-------------------+
|        06|        001|    0016|         88101|  3|